# InstaNovo+ fine-tuning — Ecoli_EV_1

Fine-tunes InstaNovo+ (diffusion) on the held-in 85% fraction of E. coli EV_1.
Saves the resulting checkpoint to Drive for the evaluation and prediction
notebooks to consume.

Mirrors `instanovo_colab_finetune.ipynb` exactly — same data, same
schedule, same patch logic — but uses `instanovo diffusion train`
instead of `instanovo transformer train` and the InstaNovo+ pretrained
ckpt.

**Inputs (must exist on Drive — sync from your project before running):**
- `MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.train.mgf`
- `MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.val.mgf`
- `MyDrive/DL-Project/bin/instanovoplus/instanovoplus-v1.1.0.ckpt`
- `MyDrive/DL-Project/config/finetune/instanovoplus.yaml`  (the Hydra config)

**Output (written to Drive):**
- `MyDrive/DL-Project/model_finetune/instanovoplus/`  (fine-tuned checkpoint + logs)

**Schedule (300 steps, ~30 steps/epoch → ~10 epochs):**
- Epochs 0–1 (steps 0–59): head + embedding only (LR warmup over first 30 steps)
- Epochs 2–4 (steps 60–149): decoder added
- Epochs 5–9 (steps 150–299): everything trains

**Vocab strategy (first run = diagnostic):** the YAML deliberately does
NOT set `override residues:` yet — we don't know whether the v1.1.0
InstaNovo+ ckpt uses the package's default 32-residue vocab or
something narrower (like InstaNovo v1.2.0's 30 residues). The patch
cell below prints the ckpt's actual residue count + keys. If they
match the package default, no override is needed. If they don't,
write a matching `config/residues/<name>.yaml` and add
`- override residues: <name>` to the YAML's defaults list.

Held-out test (`Ecoli_EV_2`) is *not* loaded here — that's consumed by the
evaluate / predict notebooks downstream.


In [1]:
!nvidia-smi

Sat May  2 16:41:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install dependencies

In [2]:
try:
  import instanovo
  !instanovo version
except ImportError:
  !pip install "instanovo[cu126]>=1.2.2" pyopenms-viz
  print('Installation complete. Restarting runtime to apply changes...')
  import os
  os.kill(os.getpid(), 9)

┏━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Package    ┃ Version ┃
┡━━━━━━━━━━━━╇━━━━━━━━━┩
│ InstaNovo  │ 1.2.2   │
│ InstaNovo+ │ 1.2.2   │
│ NumPy      │ 2.2.6   │
│ PyTorch    │ 2.8.0   │
└────────────┴─────────┘


## Sync inputs from Drive

Pulls the train/val annotated MGFs, the pretrained checkpoint, and the
Hydra config out of Drive into Colab's local filesystem.

**About the Hydra config path:** InstaNovo's CLI requires `--config-path`
to be *relative* (Hydra's `initialize()` API doesn't accept absolute paths
like `/content/config/...`). The cleanest workaround is to drop our config
into InstaNovo's own `configs/` directory — then we can pass just `-cn`
(no `-cp`) and Hydra finds it on the default search path.

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, instanovo
os.makedirs('/content/data_mgf_annotated/ecoli', exist_ok=True)
os.makedirs('/content/bin/instanovoplus', exist_ok=True)
os.makedirs('/content/config/finetune', exist_ok=True)
os.makedirs('/content/config/residues', exist_ok=True)
os.makedirs('/content/model_finetune/instanovoplus', exist_ok=True)

# Annotated train + val MGFs (same as InstaNovo finetune — same data)
!cp /content/drive/MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.train.mgf /content/data_mgf_annotated/ecoli/
!cp /content/drive/MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.val.mgf   /content/data_mgf_annotated/ecoli/

# Pretrained InstaNovo+ checkpoint from Drive — adjust SOURCE filename if yours differs.
# (Run `!ls /content/drive/MyDrive/DL-Project/bin/instanovoplus/` first if unsure.)
!cp /content/drive/MyDrive/DL-Project/bin/instanovoplus/instanovoplus-v1.1.0.ckpt /content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt

# Hydra configs — copy into InstaNovo's package configs/ dir so Hydra finds
# them via -cn / defaults (relative-path requirement). Also keep /content/
# copies for inspection in the next cell.
package_configs = os.path.join(os.path.dirname(instanovo.__file__), 'configs')
print('Package configs dir:', package_configs)

# Main finetune config
!cp /content/drive/MyDrive/DL-Project/config/finetune/instanovoplus.yaml /content/config/finetune/instanovoplus.yaml
!cp /content/drive/MyDrive/DL-Project/config/finetune/instanovoplus.yaml {package_configs}/instanovoplus_ecoli_finetune.yaml

# Custom residues file — 30-residue vocab matching v1.1.0 InstaNovo+ ckpt
# (verified: same as v1.2.0 InstaNovo, paired models share vocab). Same
# file is also referenced by config/finetune/instanovo.yaml.
# Referenced via `defaults: - override residues: 30aa` in our finetune YAML.
!cp /content/drive/MyDrive/DL-Project/config/residues/30aa.yaml /content/config/residues/30aa.yaml
!cp /content/drive/MyDrive/DL-Project/config/residues/30aa.yaml {package_configs}/residues/30aa.yaml

print('--- inputs ---')
!ls -lh /content/data_mgf_annotated/ecoli/
!ls -lh /content/bin/instanovoplus/
!ls -lh {package_configs}/instanovoplus_ecoli_finetune.yaml
!ls -lh {package_configs}/residues/30aa.yaml


Mounted at /content/drive
Package configs dir: /usr/local/lib/python3.12/dist-packages/instanovo/configs
--- inputs ---
total 66M
-rw------- 1 root root  27M May  2 16:41 Ecoli_EV_1.instanovo.train.mgf
-rw------- 1 root root 5.0M May  2 16:41 Ecoli_EV_1.instanovo.val.mgf
-rw------- 1 root root  34M May  2 16:34 Ecoli_EV_2.instanovo.annotated.mgf
total 659M
-rw------- 1 root root 659M May  2 16:41 instanovoplus-v1.1.0.ckpt
-rw------- 1 root root 3.7K May  2 16:41 /usr/local/lib/python3.12/dist-packages/instanovo/configs/instanovoplus_ecoli_finetune.yaml
-rw------- 1 root root 1.5K May  2 16:41 /usr/local/lib/python3.12/dist-packages/instanovo/configs/residues/30aa.yaml


## Patch the ckpt: wrap residues for trainer compatibility

Same root cause as InstaNovo (transformer): InstaNovo's `load_model_state`
(shared trainer base class) reads `model_residues` from
`ckpt["residues"]["residues"]` (two levels nested), but the released
`instanovoplus-v1.1.0.ckpt` stores residues flat at `ckpt["residues"]`
(one level). Without the wrap, the trainer fires
`_update_vocab(resolution="delete")` and head/embedding get stripped
from the loaded state.

This cell wraps to nested form once. It also prints the residue count
and key set so you can spot a vocab mismatch — verified on the v1.1.0
ckpt: 30 residues identical to InstaNovo v1.2.0, hence both finetune
configs share `config/residues/30aa.yaml`.

In [4]:
import torch
from omegaconf import DictConfig, OmegaConf

ckpt_path = '/content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

residues = ckpt.get('residues')

# Normalize: the v1.2.0 InstaNovo ckpt had residues as an OmegaConf DictConfig
# (not plain dict) holding 30 residues flat. Same handling here in case
# v1.1.0 is structured the same way.
if isinstance(residues, DictConfig):
    residues = OmegaConf.to_container(residues, resolve=True)

if isinstance(residues, dict):
    if 'residues' in residues:
        # Already nested — make sure inner is plain dict
        inner = residues['residues']
        if isinstance(inner, DictConfig):
            inner = OmegaConf.to_container(inner, resolve=True)
            ckpt['residues'] = {'residues': inner}
            torch.save(ckpt, ckpt_path)
            print(f'Re-saved with {len(inner)} residues nested + normalized to plain dict.')
        else:
            print(f'Ckpt already wrapped: {len(inner)} residues nested correctly.')
        flat_for_inspect = inner
    else:
        # Flat — wrap it so load_model_state can find ckpt["residues"]["residues"]
        n = len(residues)
        print(f'Wrapping {n} flat residue entries in a `residues` sub-dict.')
        ckpt['residues'] = {'residues': residues}
        torch.save(ckpt, ckpt_path)
        print(f'Saved patched ckpt to {ckpt_path}')
        flat_for_inspect = residues
    # Diagnostic: print residue count and keys so user can spot vocab mismatch
    print(f'\nv1.1.0 ckpt residue count: {len(flat_for_inspect)}')
    print(f'v1.1.0 ckpt residue keys: {list(flat_for_inspect.keys())}')
    print(f'\nIf this differs from your `override residues:` config (currently 30 residues from 30aa.yaml),')
    print(f'create a matching residues/<name>.yaml and update config/finetune/instanovoplus.yaml.')
else:
    print(f'Unexpected residues structure: type={type(residues)} value={residues!r}')
    print('Manual inspection needed before training.')


Wrapping 30 flat residue entries in a `residues` sub-dict.
Saved patched ckpt to /content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt

v1.1.0 ckpt residue count: 30
v1.1.0 ckpt residue keys: ['G', 'A', 'S', 'P', 'V', 'T', 'C', 'L', 'I', 'N', 'D', 'Q', 'K', 'E', 'M', 'H', 'F', 'R', 'Y', 'W', 'M[UNIMOD:35]', 'C[UNIMOD:4]', 'N[UNIMOD:7]', 'Q[UNIMOD:7]', 'S[UNIMOD:21]', 'T[UNIMOD:21]', 'Y[UNIMOD:21]', '[UNIMOD:1]', '[UNIMOD:5]', '[UNIMOD:385]']

If this differs from your `override residues:` config (currently 30 residues from 30aa.yaml),
create a matching residues/<name>.yaml and update config/finetune/instanovoplus.yaml.


## Verify the synced config

Quick sanity check that the config landed in Colab unchanged from your
project's `config/finetune/instanovoplus.yaml`. Skip if you trust the sync.

In [5]:
!cat /content/config/finetune/instanovoplus.yaml

defaults:
  - instanovoplus
  - finetune: default
  - override residues: 30aa   # 30-residue vocab — verified to match v1.1.0 InstaNovo+ ckpt (same as v1.2.0 InstaNovo, paired models). `override` because the `instanovoplus` parent already includes `residues: default`.
  - _self_

# Fine-tuning trigger — points at the pretrained InstaNovo+ ckpt copied from Drive.
resume_checkpoint_path: /content/bin/instanovoplus/instanovoplus-v1.1.0.ckpt

# Fine-tuning hyperparameters (small data -> low LR, short schedule).
# ~960 train spectra / 32 batch ~= 30 steps/epoch x 10 epochs ~= 300 steps.
learning_rate: 5.0e-6
warmup_iters: 30          # 1 epoch warmup (LR ramps 0 -> 5e-6)
training_steps: 300
train_batch_size: 32
validation_interval: 30
checkpoint_interval: 30
console_logging_steps: 10
tensorboard_logging_steps: 10
model_save_folder_path: /content/model_finetune/instanovoplus

# Track best ckpt by lowest valid_loss — these fields are NOT in the
# diffusion parent config (the transformer paren

## Run fine-tuning

Hydra resolves `instanovoplus_ecoli_finetune.yaml` from InstaNovo's
package configs directory (where we copied it), composes it on top of
the default `instanovoplus` + `finetune: default` configs, and trains
via the diffusion entrypoint.

The CLI overrides on the next cell delete the same five InstaNovo-internal
validation entries that Hydra dict-merges in by default (same merge
behavior as the transformer config — different command, same dataset
config).

Three configuration choices baked into the YAML, worth knowing about
if you ever modify it:

- **`dataset.lazy_loading: true`** — required. The trainer's
  `.shuffle(buffer_size=...)` call only works on `IterableDataset`,
  which lazy mode produces. With `lazy_loading: false` the dataset is
  a regular `Dataset` and `.shuffle()` rejects `buffer_size`.
- **`finetune.unfreeze_format: start_step`** (not `start_epoch`) —
  the trainer doesn't pass `steps_per_epoch` to `FinetuneScheduler`,
  so any epoch-based schedule raises a ValueError. Our step boundaries
  (0 / 60 / 150) encode the same 0 / 2 / 5-epoch schedule.
- **Layer names differ from transformer** — head is
  `transition_model.head.1.{weight,bias}`, embedding is
  `transition_model.char_embedding.weight`, decoder is
  `transition_model.decoder.*`. If the train log shows
  `Unfrozen 0 parameters in 0 layers` for any phase, double-check
  these names against the actual model state_dict keys.

In [6]:
# Hydra/OmegaConf merges dicts (it doesn't replace them) when composing
# defaults + our YAML. Our `dataset.valid_path.ecoli: ...` got *added* to
# the default dict's 5 InstaNovo-internal parquet entries instead of
# replacing them, so the loader then tries to read those (non-existent)
# files. Use `~key.path` to delete them at the CLI.
!instanovo diffusion train -cn instanovoplus_ecoli_finetune \
    ~dataset.valid_path.acpt \
    ~dataset.valid_path.phospho \
    ~dataset.valid_path.pride \
    ~dataset.valid_path.massivekb \
    ~dataset.valid_path.lcfm

[05/02/26 16:41:14] INFO     Initializing InstaNovo+ training.                                                                                                                 
[05/02/26 16:41:16] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 16:41:19.203824: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 16:41:19.274952: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 16:41

## Push fine-tuned checkpoint + logs to Drive

Other notebooks (evaluate / predict) read from this Drive path. Re-running
this cell after a fresh fine-tune overwrites previous artifacts.

In [7]:
!mkdir -p /content/drive/MyDrive/DL-Project/model_finetune/instanovoplus
!cp -r /content/model_finetune/instanovoplus/. /content/drive/MyDrive/DL-Project/model_finetune/instanovoplus/
!ls -lh /content/drive/MyDrive/DL-Project/model_finetune/instanovoplus/

total 1.3G
-rw------- 1 root root 659M May  2 16:45 model_best.ckpt
-rw------- 1 root root 659M May  2 16:45 model_latest.ckpt
